### 环境配置
MindSpore 2.3
MindNLP 0.3.1
Python 3.9

In [ ]:
%%capture captured_output
!/home/ma-user/anaconda3/bin/conda create -n python-3.9.0 python=3.9.0 -y --override-channels --channel https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
!/home/ma-user/anaconda3/envs/python-3.9.0/bin/pip install ipykernel

In [ ]:
import json
import os

data = {
   "display_name": "python-3.9.0",
   "env": {
      "PATH": "/home/ma-user/anaconda3/envs/python-3.9.0/bin:/home/ma-user/anaconda3/envs/python-3.7.10/bin:/modelarts/authoring/notebook-conda/bin:/opt/conda/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/ma-user/modelarts/ma-cli/bin:/home/ma-user/modelarts/ma-cli/bin"
   },
   "language": "python",
   "argv": [
      "/home/ma-user/anaconda3/envs/python-3.9.0/bin/python",
      "-m",
      "ipykernel",
      "-f",
      "{connection_file}"
   ]
}

if not os.path.exists("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/"):
    os.mkdir("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/")

with open('/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/kernel.json', 'w') as f:
    json.dump(data, f, indent=4)

安装完成后重启kernel，选择python3.9

In [ ]:
%%capture captured_output
!pip uninstall mindspore-gpu -y
!pip install https://ms-release.obs.cn-north-4.myhuaweicloud.com/2.3/MindSpore/unified/x86_64/mindspore-2.3-cp39-cp39-linux_x86_64.whl --trusted-host ms-release.obs.cn-north-4.myhuaweicloud.com -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install download nltk -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install mindnlp

### 加载数据
首先，加载图像数据集

In [ ]:
from datasets import load_dataset
# 数据来源：https://github.com/phelber/EuroSAT
dataset = load_dataset("imagefolder", data_files="EuroSAT_RGB.zip")

图像分类任务数据通常包含两类数据：图像和标签

In [ ]:
dataset["train"].features

我们首先可视化一个示例

In [ ]:
example = dataset["train"][0]
example["image"]

In [ ]:
example["label"]

当前标签展示为整数，我们可以通过如下的方式将它们转化为真实的分类名

In [ ]:
labels = dataset["train"].features["label"].names
labels

In [ ]:
id2label = {k:v for k,v in enumerate(labels)}
id2label

上图图像的标签为'AnnualCrop'

In [ ]:
id2label[0]

### 加载模型
从hub加载模型和处理器

In [ ]:
from mindnlp.transformers.models import ConvNextForImageClassification,ConvNextImageProcessor
img_processor = ConvNextImageProcessor.from_pretrained("nielsr/convnext-tiny-finetuned-eurosat")
model = ConvNextForImageClassification.from_pretrained("nielsr/convnext-tiny-finetuned-eurosat")

我们取前边的例子进行测试

In [ ]:
test_image = example["image"].convert("RGB")
test_image

### 图像预处理
使用处理器对测试图像进行预处理，将其转换为张量，并提取预处理后的像素值

In [ ]:
pixel_values = img_processor(test_image,return_tensors="ms").pixel_values
pixel_values.shape

### 前向传播
将预处理后的像素值输入到模型中进行前向传播，提取模型输出的logits

In [ ]:
outputs = model(pixel_values)
logits = outputs.logits
logits.shape

### 预测类别
根据logits得分，获取预测类别。

In [ ]:
predicted_class_idx = logits.argmax(-1).item()
model.config.id2label[predicted_class_idx]